# Bulk method (refactored)

This notebook refactors the original `Bulk_method.ipynb` with a clearer structure while preserving the scientific calculations and variable units.


## 1) Data loading

Load libraries, configure plotting defaults, define model/configuration, and open CMIP6 variables.


In [ ]:
import os
import sys
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import intake
import gsw

sys.path.append('/g/data/jk72/zc0441')
from zpackage import wmt
from zpackage.ztake import *


In [ ]:
# Publication-oriented plotting defaults (consistent figure style)
plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams.update({
    'figure.figsize': (9, 4.8),
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'savefig.dpi': 150,
})


In [ ]:
# Core configuration
base_data_dir = '/g/data/jk72/zc0441/Subject_3/data/'
time_range = slice('1979', '2014')
model = 'ACCESS-ESM1-5'

print(f'Model: {model}')
print(f'Time range: {time_range.start} to {time_range.stop}')


In [ ]:
# Load CMIP catalogs and initialize Ztake handlers used below
cmip6 = intake.cat.access_nri['cmip6_oi10']
cmip6_fs38 = intake.cat.access_nri['cmip6_fs38']

def make_hist_ztake(variable_id):
    constraints = dict(
        experiment_id='historical',
        variable_id=variable_id,
        member_id=['r1i1p1f1', 'r2i1p1f1', 'r1i1p1f2'],
        table_id='Omon',
    )
    return Ztake(
        cmip6_catalog=cmip6_fs38,
        constraints=constraints,
        prefer_members=('r1i1p1f1', 'r1i1p1f2'),
        prefer_grids=('gn', 'gr', 'gr1'),
    )

zt_fgco2_hist_fs38 = make_hist_ztake('fgco2')
zt_dissic_hist_fs38 = make_hist_ztake('dissic')
zt_wfo_hist_fs38 = make_hist_ztake('wfo')
zt_hfds_hist_fs38 = make_hist_ztake('hfds')
zt_sos_hist_fs38 = make_hist_ztake('sos')
zt_tos_hist_fs38 = make_hist_ztake('tos')


In [ ]:
# Keep piControl hfds Ztake initialization from the original workflow
constraints = dict(
    experiment_id='piControl',
    variable_id='hfds',
    member_id=['r1i1p1f1', 'r2i1p1f1', 'r1i1p1f2'],
    table_id='Omon',
)

zt_hfds_pi = Ztake(
    cmip6_catalog=cmip6,
    constraints=constraints,
    prefer_members=('r1i1p1f1', 'r1i1p1f2'),
    prefer_grids=('gn', 'gr', 'gr1'),
)

zt_hfds_pi_fs38 = Ztake(
    cmip6_catalog=cmip6_fs38,
    constraints=constraints,
    prefer_members=('r1i1p1f1', 'r1i1p1f2'),
    prefer_grids=('gn', 'gr', 'gr1'),
)


In [ ]:
# Open core variables from the same sources used in the original notebook
fgco2 = zt_fgco2_hist_fs38.open_model(model)['fgco2'].sel(time=time_range)
dissic = zt_dissic_hist_fs38.open_model(model)['dissic'].sel(time=time_range)
wfo = zt_wfo_hist_fs38.open_model(model)['wfo'].sel(time=time_range)
hfds = zt_hfds_hist_fs38.open_model(model)['hfds'].sel(time=time_range)
sos = zt_sos_hist_fs38.open_model(model)['sos'].sel(time=time_range)
tos = zt_tos_hist_fs38.open_model(model)['tos'].sel(time=time_range)

# Static area (historical areacello file convention preserved)
area_path = f"{base_data_dir}/{model}/areacello_historical_gn.nc"
areacello = xr.open_dataset(area_path)['areacello']


## 2) Preprocessing

Standardize dimension names, build reusable helper functions, and prepare blocked/annual aggregates.


In [ ]:
def get_i_j_dim(da):
    """Return horizontal dims (i/j-like) excluding known non-horizontal dimensions."""
    exclude = {'time', 'year', 'month', 'lev', 'depth', 'z', 'sigma', 'rho', 'density', 'olevel', 'block'}
    spatial = [d for d in da.dims if d not in exclude]
    if len(spatial) != 2:
        raise ValueError(f'Expected 2 horizontal dims, got {spatial}')
    return spatial[0], spatial[1]


def standardize_horizontal_dims(da, i_name='i', j_name='j'):
    """Rename horizontal dimensions to consistent names when needed."""
    i_dim, j_dim = get_i_j_dim(da)
    rename_map = {}
    if i_dim != i_name:
        rename_map[i_dim] = i_name
    if j_dim != j_name:
        rename_map[j_dim] = j_name
    return da.rename(rename_map) if rename_map else da


def decadal_block_monthly_mean(da):
    """Convert monthly time series to (block, month, ...), where each block is 10 years."""
    years = da['time'].dt.year
    block = ((years - years.min()) // 10).astype('int32').rename('block')
    return da.assign_coords(block=block).groupby('block').groupby('time.month').mean('time')


def smooth_along_density(da, win=5, center=True, min_periods=1):
    """Smooth along density bins with a rolling mean."""
    return da.rolling(density=win, center=center, min_periods=min_periods).mean()


def detrend_along_dim(da, dim):
    """Linear detrend along a specified dimension."""
    coef = da.polyfit(dim=dim, deg=1)
    trend = xr.polyval(da[dim], coef.polyfit_coefficients)
    return da - trend, trend


def tr_to_fr(transformation_rate):
    """Convert transformation rate to formation rate (logic preserved from original notebook)."""
    if not isinstance(transformation_rate, xr.DataArray):
        raise ValueError('Input must be an xarray DataArray')

    fr = xr.zeros_like(transformation_rate)

    if transformation_rate.ndim == 1:
        for index, value in enumerate(transformation_rate):
            value = value.item()
            if value > 0:
                fr[index] += abs(value)
                fr[index - 1] -= abs(value)
            else:
                fr[index - 1] += abs(value)
                fr[index] -= abs(value)
    elif transformation_rate.ndim == 2:
        for i in range(transformation_rate.shape[0]):
            for j in range(transformation_rate.shape[1]):
                value = transformation_rate[i, j].item()
                if value > 0:
                    fr[i, j] += abs(value)
                    fr[i, j - 1] -= abs(value)
                else:
                    fr[i, j - 1] += abs(value)
                    fr[i, j] -= abs(value)
    else:
        raise ValueError('Input must be a 1D or 2D xarray DataArray')

    return fr


In [ ]:
def calculate_wmt_from_blocked(
    data_dict,
    model,
    variable_name='sidd_weighted_monthly',
    density_min=1024,
    density_max=1029,
    step=0.1,
    unit='Sv',
    v_type='sea ice',
    block_dim='block',
    time_dim=None,
):
    """Compute WMT(density) per block from pre-blocked monthly fields."""
    sidd_data = data_dict[model][variable_name]
    density_data = data_dict[model]['density_monthly']
    temperature_data = data_dict[model]['tos_monthly']
    salinity_data = data_dict[model]['sos_monthly']

    if unit == 'Pg':
        Pg = 1e12
        sidd_data_in_unit = sidd_data / Pg * 365.25 * 24 * 360
    else:
        Sv = 1e6
        sidd_data_in_unit = sidd_data / Sv

    if time_dim is None:
        if 'month' in sidd_data_in_unit.dims:
            time_dim = 'month'
        elif 'time' in sidd_data_in_unit.dims:
            time_dim = 'time'
        else:
            raise ValueError(f"Cannot infer time dimension from {sidd_data_in_unit.dims}")

    lat_coord, lon_coord = get_lat_lon_coords(sidd_data_in_unit)
    sidd_data_in_unit = sidd_data_in_unit.where(sidd_data_in_unit[lat_coord] < -45)

    if 'CMCC' in model:
        density_data = density_data[..., :-1, 1:-1]
        temperature_data = temperature_data[..., :-1, 1:-1]
        salinity_data = salinity_data[..., :-1, 1:-1]

    if 'NorESM' in model and 'sid' in variable_name:
        density_data = density_data[..., :-1, :]
        temperature_data = temperature_data[..., :-1, :]
        salinity_data = salinity_data[..., :-1, :]

    if 'FGOALS' in model:
        density_data = density_data.isel(j=slice(None, None, -1))
        temperature_data = temperature_data.isel(j=slice(None, None, -1))
        salinity_data = salinity_data.isel(j=slice(None, None, -1))

    if 'CAS-ESM' in model:
        sidd_data_in_unit = sidd_data_in_unit.roll({sidd_data_in_unit.dims[-1]: 1}, roll_coords=True)

    density_centers = np.arange(density_min, density_max, step)
    block_vals = sidd_data_in_unit[block_dim].values
    results = []

    for b in block_vals:
        sidd_sl = sidd_data_in_unit.sel({block_dim: b})
        dens_sl = density_data.sel({block_dim: b})
        temp_sl = temperature_data.sel({block_dim: b})
        sal_sl = salinity_data.sel({block_dim: b})

        if v_type == 'water':
            buoy_rate = wmt.trans_rate_water(sidd_sl, sal_sl, temp_sl, p=0)
        elif v_type == 'heat':
            buoy_rate = wmt.trans_rate_heat(sidd_sl, sal_sl, temp_sl, p=0)
        elif v_type == 'sea ice':
            if ('CAS-ESM' in model) or ('CESM2' in model) or ('CIESM' in model):
                buoy_values = wmt.trans_rate_seaice(sidd_sl.values, sal_sl.values, temp_sl.values, p=0)
                buoy_rate = xr.DataArray(buoy_values, dims=sidd_sl.dims, coords=sidd_sl.coords)
            else:
                buoy_rate = wmt.trans_rate_seaice(sidd_sl, sal_sl, temp_sl, p=0)
        else:
            buoy_rate = sidd_sl

        wmt_rate = -buoy_rate / step
        ntime = wmt_rate.sizes[time_dim]

        block_curve = []
        for center in density_centers:
            lower = center - step / 2
            upper = center + step / 2
            mask = (dens_sl >= lower) & (dens_sl < upper)
            if ('CAS-ESM' in model) or ('CESM2' in model) or ('CIESM' in model):
                acc = wmt_rate.where(mask.values).sum()
            else:
                acc = wmt_rate.where(mask).sum()
            block_curve.append(acc / ntime)

        results.append(xr.DataArray(block_curve, coords={'density': density_centers}, dims=['density']))

    out = xr.concat(results, dim=xr.IndexVariable(block_dim, block_vals))
    out.attrs['note'] = f'Computed per {block_dim}; normalized by n({time_dim}) within each block.'
    return out


In [ ]:
# Enforce consistent horizontal dimensions for all 2D/3D fields
fgco2 = standardize_horizontal_dims(fgco2)
dissic = standardize_horizontal_dims(dissic)
wfo = standardize_horizontal_dims(wfo)
hfds = standardize_horizontal_dims(hfds)
sos = standardize_horizontal_dims(sos)
tos = standardize_horizontal_dims(tos)
areacello = standardize_horizontal_dims(areacello)

# Build area-weighted fields used later in the same way as the original notebook
fgco2_area = fgco2 * areacello
wfo_area = wfo * areacello
hfds_area = hfds * areacello

# Annual Southern Ocean integrated fgco2 (45°S and south)
lat_coord, lon_coord = get_lat_lon_coords(fgco2_area)
so_mask = fgco2_area[lat_coord] < -45
fgco2_area_mean_by_year = fgco2_area.where(so_mask).groupby('time.year').mean('time').sum(['j', 'i'])

# Decadal monthly blocks for WMT workflow
tos_dec = decadal_block_monthly_mean(tos).compute()
sos_dec = decadal_block_monthly_mean(sos).compute()
hfds_dec = decadal_block_monthly_mean(hfds_area).compute()
wfo_dec = decadal_block_monthly_mean(wfo_area).compute()

# Surface density (same formula as original)
density_dec = gsw.rho(sos_dec, tos_dec, 0)


## 3) Analysis

Compute WMT/WMF components from blocked fields using unchanged physics and conversion logic.


In [ ]:
# Data dictionary expected by calculate_wmt_from_blocked
data_dict = {
    model: {
        'tos_monthly': tos_dec,
        'sos_monthly': sos_dec,
        'density_monthly': density_dec,
        'hfds_area_monthly': hfds_dec,
        'wfo_area_monthly': wfo_dec,
    }
}

# WMT components (heat + freshwater), matching original sign conventions
wmt_hfds_block = -calculate_wmt_from_blocked(data_dict, model, variable_name='hfds_area_monthly', v_type='heat')
wmt_wfo_block = -calculate_wmt_from_blocked(data_dict, model, variable_name='wfo_area_monthly', v_type='water')

# Smoothed and total WMT/WMF
wmt_hfds_block_s = smooth_along_density(wmt_hfds_block, win=5)
wmt_wfo_block_s = smooth_along_density(wmt_wfo_block, win=5)

wmt_total_block = wmt_wfo_block + wmt_hfds_block
wmt_total_block_s = wmt_wfo_block_s + wmt_hfds_block_s
wmf_total_block_s = smooth_along_density(tr_to_fr(wmt_total_block_s), win=5)


## 4) Visualization

Use consistent color maps, larger labels, and explicit plot descriptions for publication-ready figures.


In [ ]:
def plot_wmt_blocks_colored_by_year(
    wmt_block,
    block_dim='block',
    density_dim='density',
    block_years=10,
    start_year=1979,
    cmap='viridis',
    ax=None,
    lw=2.0,
):
    """Plot WMT(density) for each block with line color indicating block start year."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 5))
    else:
        fig = ax.figure

    bvals = wmt_block[block_dim].values
    years = (start_year + bvals.astype(int) * int(block_years)).astype(int)

    norm = mpl.colors.Normalize(vmin=years.min(), vmax=years.max())
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)

    dens = wmt_block[density_dim].values
    for bi, yr in zip(bvals, years):
        y = wmt_block.sel({block_dim: bi}).values
        ax.plot(dens, y, color=sm.to_rgba(yr), lw=lw, alpha=0.9)

    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label('Block start year')
    ax.set_title('Water Mass Transformation by density and decade')
    ax.set_xlabel('Potential density (kg m$^{-3}$)')
    ax.set_ylabel('Transformation (Sv)')
    return ax


In [ ]:
# Plot 1: Southern Ocean annual integrated air-sea CO2 flux
# This figure shows interannual variability of integrated fgco2 south of 45°S.
fig, ax = plt.subplots(figsize=(10, 4.5))
fgco2_area_mean_by_year.plot(ax=ax, color='tab:blue', lw=2)
ax.set_title(f'{model}: Southern Ocean integrated fgco2 (south of 45°S)')
ax.set_xlabel('Year')
ax.set_ylabel('Integrated fgco2 (native model units × area)')
plt.tight_layout()


In [ ]:
# Plot 2: Smoothed total WMT curves by decade
# Each line is one decadal block; color encodes block start year.
fig, ax = plt.subplots(figsize=(10, 5.2))
plot_wmt_blocks_colored_by_year(
    wmt_total_block_s,
    start_year=int(fgco2['time'].dt.year.min()),
    block_years=10,
    cmap='cividis',
    ax=ax,
)
ax.axhline(0, color='k', lw=0.8)
plt.tight_layout()


In [ ]:
# Plot 3: WMT component comparison (block mean)
# This panel compares heat-driven and freshwater-driven contributions to mean WMT.
fig, ax = plt.subplots(figsize=(10, 5.2))
wmt_hfds_block_s.mean('block').plot(ax=ax, color='tab:red', lw=2, label='Heat component')
wmt_wfo_block_s.mean('block').plot(ax=ax, color='tab:green', lw=2, label='Freshwater component')
wmt_total_block_s.mean('block').plot(ax=ax, color='tab:blue', lw=2.5, label='Total')
ax.set_title(f'{model}: Mean WMT components across blocks')
ax.set_xlabel('Potential density (kg m$^{-3}$)')
ax.set_ylabel('Transformation (Sv)')
ax.axhline(0, color='k', lw=0.8)
ax.legend(frameon=True)
plt.tight_layout()
